
# Assignment 1: Boolean Model, TF-IDF, and Data Retrieval vs. Information Retrieval Conceptual Questions

**Student names**: Eskild Sæther <br>
**Group number**: 101 <br>
**Date**: 9.9.2026

## Important notes
Please carefully read the following notes and consider them for the assignment delivery. Submissions that do not fulfill these requirements will not be assessed and should be submitted again.
1. You may work in groups of maximum 2 students.
2. The assignment must be delivered in ipynb format.
3. The assignment must be typed. Handwritten assignments are not accepted.

**Due date**: 18.09.2026 23:59

In this assignment, you will:
- Implement a Boolean retrieval model
- Compute TF-IDF vectors for documents
- Run retrieval on queries
- Answer conceptual questions 

---
## Dataset

You will use the **Cranfield** dataset, provided in this file:

- `cran.all.1400`: The document collection (1400 documents)

**The code to parse the file is ready — just update the cran file path to match your own file location. Use the docs variable in your code for the parsed file**

### Load and parse documents (provided)

Run the cell to parse the Cranfield documents. Update the path so it points to your `cran.all.1400` file.


In [1]:

# Read 'cran.all.1400' and parse the documents into a suitable data structure

CRAN_PATH = r"./cran.all.1400"  # <-- change this!

def parse_cranfield(path):
    docs = {}
    current_id = None
    current_field = None
    buffers = {"T": [], "A": [], "B": [], "W": []}
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith(".I "):
                if current_id is not None:
                    docs[current_id] = {
                        "id": current_id,
                        "title": " ".join(buffers["T"]).strip(),
                        "abstract": " ".join(buffers["W"]).strip()
                    }
                current_id = int(line.split()[1])
                buffers = {k: [] for k in buffers}
                current_field = None
            elif line.startswith("."):
                tag = line[1:].strip()
                current_field = tag if tag in buffers else None
            else:
                if current_field is not None:
                    buffers[current_field].append(line)
    if current_id is not None:
        docs[current_id] = {
            "id": current_id,
            "title": " ".join(buffers["T"]).strip(),
            "abstract": " ".join(buffers["W"]).strip()
        }
    print(f"Parsed {len(docs)} documents.")
    return docs

docs = parse_cranfield(CRAN_PATH)

#litt python dicts ig
print(f"Information about the first document \n{docs[1]['title']}")



Parsed 1402 documents.
Information about the first document 
Experimental investigation of the aerodynamics of a wing in a slipstream .


## 1.1 – Boolean Retrieval Model

### 1.1.1 Tokenize documents

Implement tokenization using the given list of stopwords. Create a list of normalized terms per document (e.g., lowercase, remove punctuation/digits; drop stopwords). Store the token lists to use in later steps.

In [2]:
# TODO: Implement tokenization using the given list of stopwords, create list of terms per document

STOPWORDS = set("""a about above after again against all am an and any are aren't as at be because been
before being below between both but by can't cannot could couldn't did didn't do does doesn't doing don't down
during each few for from further had hadn't has hasn't have haven't having he he'd he'll he's her here here's hers
herself him himself his how how's i i'd i'll i'm i've if in into is isn't it it's its itself let's me more most
mustn't my myself no nor not of off on once only or other ought our ours ourselves out over own same shan't she
she'd she'll she's should shouldn't so some such than that that's the their theirs them themselves then there there's
these they they'd they'll they're they've this those through to too under until up very was wasn't we we'd we'll we're
we've were weren't what what's when when's where where's which while who who's whom why why's with won't would wouldn't
you you'd you'll you're you've your yours yourself yourselves""".split())

# Your code here
#Skal fjerne alle stopwords fra documentene
#Lager en ny key (terms) som inneholder terms, må kombinere tittel og abstract
#Det virker som datasettet bare er lowercase, men jeg kan uansett gjøre alt lowervase for å få litt python trening

#importerer regex (nå har jeg 2 problemer elns)
import re

for document in docs.values():
    for key, value in document.items():
        if(type(value)==str):
            document[key] = value.lower()
            #print(document[key])

            #sub = substitute, misliker så mange forkortelser, r betyr at python ikke rører \x type symboler
            document[key] = re.sub(r"[.,/+\()\d\\]", "", document[key])

for document in docs.values():
    terms = []
    for word in document["title"].split():
        if(word not in STOPWORDS):
            terms.append(word)

    for word in document["abstract"].split():
        if(word not in STOPWORDS):
            terms.append(word)
    
    #terms = terms.union(set(document["title"].split()).difference(STOPWORDS))
    #terms = terms.union(set(document["abstract"].split()).difference(STOPWORDS))

    document["terms"] = terms



### Build vocabulary

Create a set (or list) of unique terms from all tokenized documents. Report the number of unique terms.


In [3]:
# TODO: Create a set or list of unique terms

# Report: 
# - Number of unique terms

# Your code here
unique_terms = set()
for document in docs.values():
    unique_terms = unique_terms.union(document["terms"])

print(unique_terms, "\n", len(unique_terms))


{'similar', 'r=', 're-emphasizes', 'housefly', 'deficit', 'claim', 'uncertain', 'electronic', 'hovercraft', 'conduction-controlled', 'practicably', 'secondary-flow', 'relay', 'transit', 'conclusion', 'corollary', 'geometrics', 'isobar', 'blades', 'landing', 'icbm', 'kelly', 'exhaustive', 'zones', "graham's", 'wedge-flow', 'correlator', 'specializations', 'thoroughly', 'formations', 'whatever', 'constant-area', 'viscosity-temperature', 'lateraldirectional', 'uniformly', 'by-', 'justifies', 'severe', 'incident', 'epoxy', 'gravity', 'schoenherr', 'scales', 'assessed', 'pressure-drag', 'predominantly', 'brief', 'treatment', 'today', 'irrespective', 'e-versus-time', 'realistic', 'efficiencies', 'individual', 'engineer', 'delimited', 'press', 'contracts', 'evaporates', 'translational', 'gradual', 'cross-drift', 'nomenclature', 'four-wave', 'aspect-ratio', 'wing-fin', 'five-stage', 'taken', 'usable', 'reciprocal', 'high-altitude', 'vectors', 'figures', 'horizontally', 'hitherto', 'prevented',

### Build inverted index

For each term, store the list (or set) of document IDs where the term appears.


In [4]:

# TODO: For each term, store list of document IDs where the term appears
# Your code here

inverted_index = {}

for term in unique_terms:
    inverted_index[term] = []

for document in docs.values():
    for term in document['terms']:
        if(document["id"] not in inverted_index[term]):
            inverted_index[term].append(document['id'])

print(inverted_index)


{'similar': [11, 13, 23, 25, 44, 51, 62, 72, 73, 77, 90, 94, 96, 97, 101, 104, 107, 118, 140, 147, 156, 160, 173, 177, 192, 199, 204, 227, 229, 252, 262, 267, 292, 293, 300, 305, 308, 309, 315, 335, 371, 417, 435, 446, 449, 453, 473, 491, 495, 506, 526, 538, 540, 560, 562, 565, 567, 576, 581, 583, 629, 638, 641, 657, 677, 682, 683, 698, 701, 721, 726, 733, 757, 764, 811, 812, 822, 826, 871, 904, 928, 937, 941, 972, 976, 1007, 1037, 1040, 1059, 1072, 1101, 1103, 1106, 1110, 1149, 1175, 1181, 1200, 1206, 1219, 1235, 1242, 1246, 1263, 1271, 1280, 1399], 'r=': [417, 654], 're-emphasizes': [526], 'housefly': [933], 'deficit': [309, 1288], 'claim': [1310], 'uncertain': [640], 'electronic': [66, 93, 498, 704, 869, 878, 1113, 1280, 1286], 'hovercraft': [649, 650], 'conduction-controlled': [85], 'practicably': [866], 'secondary-flow': [660], 'relay': [367], 'transit': [395], 'conclusion': [35, 43, 82, 83, 152, 156, 195, 373, 497, 534, 536, 557, 618, 625, 629, 640, 644, 792, 802, 860, 872, 1025,

### Retrieve documents for a Boolean query (AND/OR)

Create a function to retrieve documents for a Boolean query (AND/OR) with query terms.  


In [5]:
# TODO: Create a function for retrieving documents for a Boolean query (AND/OR) with query terms

def boolean_retrieve(query:str) -> set[int]: 
# Your code here
    #litt morsom string manipulering
    documentsIds: set(int) = set()

    #Veldig forvirra på operation precedence, men tar det bare i kronologisk rekkefølge.
    #Vil ha det nærmest formatert so men stack.
    arguments = query.split()

    #Kansje det er enklere å ta 2 og 2 faktisk.
    documentsIds = set(inverted_index[arguments[0]])
    arguments = arguments[1:]

    while (len(arguments) >= 2):
        if(arguments[0] == "AND"):
            for id in documentsIds.copy():
                if(arguments[1] not in docs[id]['terms']):
                    documentsIds.discard(id)

        elif(arguments[0] == "OR"):
            documentsIds.union(set(inverted_index[arguments[1]]))

        arguments = arguments[2:]


    return list(documentsIds)

boolean_retrieve("gas AND flight AND flow")


[33, 49, 574, 85, 625, 1295, 1310, 319, 456]

In [6]:
# Do not change this code
boolean_queries = [
  "gas AND pressure",
  "structural AND aeroelastic AND flight AND high AND speed OR aircraft",
  "heat AND conduction AND composite AND slabs",
  "boundary AND layer AND control",
  "compressible AND flow AND nozzle",
  "combustion AND chamber AND injection",
  "laminar AND turbulent AND transition",
  "fatigue AND crack AND growth",
  "wing AND tip AND vortices",
  "propulsion AND efficiency"
]

In [7]:
# Run Boolean queries in batch, using the function you created
def run_batch_boolean(queries):
    results = {}
    for i, q in enumerate(queries, 1):
        res = boolean_retrieve(q)
        results[f"Q{i}"] = res
    return results

boolean_results = run_batch_boolean(boolean_queries)
for qid, res in boolean_results.items():
    print(qid, "=>", res[:5])


Q1 => [518, 529, 536, 27, 540]
Q2 => [12]
Q3 => [5, 399]
Q4 => [244, 265, 342, 792, 798]
Q5 => [118, 131]
Q6 => []
Q7 => [7, 9, 522, 526, 536]
Q8 => []
Q9 => [675]
Q10 => [968]


## Part 1.2 – TF-IDF Indexing


$tf_{i,j} = \text{Raw Frequency}$

$idf_t = \log\left(\frac{N}{df_t}\right)$

### Build document–term matrix (TF and IDF weights)

Compute tf and idf using the formulas above and store the weights in a document–term matrix (rows = documents, columns = terms).



In [8]:
# TODO: Calculate the weights for the documents and the terms using tf and idf weighting. Put these values into a document–term matrix (rows = documents, columns = terms).

# Your code here
import math

tf = [[]]

for document in docs.values():
    document['tf'] = {}
    for term in document['terms']:
        document['tf'][term] = document['terms'].count(term)

print(docs[1]['tf'])

#Document frequency
idf = {}
for term in unique_terms:
    idf[term] = len(inverted_index[term])
    idf[term] = math.log(len(docs)/idf[term], 10)

print(idf)
        

{'experimental': 3, 'investigation': 2, 'aerodynamics': 2, 'wing': 4, 'slipstream': 6, 'study': 1, 'propeller': 1, 'made': 2, 'order': 1, 'determine': 1, 'spanwise': 1, 'distribution': 1, 'lift': 4, 'increase': 1, 'due': 2, 'different': 3, 'angles': 1, 'attack': 1, 'free': 1, 'stream': 1, 'velocity': 1, 'ratios': 1, 'results': 1, 'intended': 1, 'part': 2, 'evaluation': 2, 'basis': 1, 'theoretical': 1, 'treatments': 1, 'problem': 1, 'comparative': 1, 'span': 1, 'loading': 1, 'curves': 1, 'together': 1, 'supporting': 1, 'evidence': 1, 'showed': 1, 'substantial': 1, 'increment': 2, 'produced': 1, 'destalling': 3, 'boundary-layer-control': 1, 'effect': 1, 'integrated': 1, 'remaining': 1, 'subtracting': 1, 'found': 1, 'agree': 1, 'well': 1, 'potential': 1, 'flow': 1, 'theory': 1, 'empirical': 1, 'effects': 1, 'specific': 1, 'configuration': 1, 'experiment': 1}
{'similar': 1.11736423594543, 'r=': 2.8457180179666586, 're-emphasizes': 3.1467480136306394, 'housefly': 3.1467480136306394, 'defici

### Build TF–IDF document vectors

From the matrix, build a TF–IDF vector for each document (consider normalization if needed for cosine similarity).


In [9]:

# TODO: Build TF–IDF document vectors from the document–term matrix
# Your code here
#TF-IDF is for each document that multiplies th term frequency with the rarity of the term. The term is the dimension
tf_idf = {}

for document in docs.values():
    document['tf-idf'] = {}
    for term in document['terms']:
        document['tf-idf'][term] = (1 + math.log(document['tf'][term], 10))*idf[term]


print(docs[1]['tf-idf'])
print(docs[2]['tf-idf'])


{'experimental': 0.9557874542968668, 'investigation': 1.0594411551390097, 'aerodynamics': 2.322364793940021, 'wing': 1.472070940755321, 'slipstream': 3.6764464330294055, 'study': 1.0006199779524019, 'propeller': 1.8914755085273338, 'made': 0.7824926478304812, 'order': 0.991411976165578, 'determine': 1.1381478418687223, 'spanwise': 1.748808004958602, 'distribution': 0.7647309710557714, 'lift': 1.6763594781318352, 'increase': 1.155521937938145, 'due': 1.289856719051921, 'different': 1.6565060332897068, 'angles': 1.173620160030941, 'attack': 1.1255587145607016, 'free': 1.1339107889254676, 'stream': 0.9944596692475832, 'velocity': 0.7042682445661913, 'ratios': 1.1467480136306398, 'results': 0.37150175389040335, 'intended': 2.0675667675830147, 'part': 1.6040843264610585, 'evaluation': 2.298317328710084, 'basis': 1.3142391009244034, 'theoretical': 0.8102882797821104, 'treatments': 2.1053553284724145, 'problem': 0.771999667620536, 'comparative': 2.368596763246996, 'span': 1.6995899822884204, 

### Implement cosine similarity

Implement a function to compute cosine similarity scores between a (tokenized) query and all documents.


In [10]:

# TODO: Create a function for calculating the similarity score of all the documents by their relevance to query terms

def tfidf_retrieve(query: str):
    # Your code here
    #skal query ha vekt? idk
    tokens = query.split()

    #skal jo ha vektor ting men er bare poeng i ta det de faktisk er tilstede.
    score = []

    normalization_factor_2 = 0
    for term in tokens:
        if(term not in unique_terms):
            tokens.remove(term)
        else:
            normalization_factor_2 += idf[term]
    
    for document in docs.values():
        similarity = 0

        for term in tokens:
            if(term in document['terms']):
                similarity += idf[term] * document['tf-idf'][term]

        normalization_factor_1 = math.sqrt(sum(weight**2 for weight in document['tf-idf'].values()))

        normalization_factor = normalization_factor_1 * normalization_factor_2

        if(normalization_factor != 0):
            similarity = (similarity/normalization_factor)
        
        output = ["ID " + str(document['id']), similarity]
        score.append(output)
        
        
    #score.sort(reverse=True)
    def sortingfunc(output):
        return output[1]

    score.sort(reverse=True, key=sortingfunc)

    return(score)

#tfidf_retrieve("function paper flight gas")


In [11]:
# Do not change this code
tfidf_queries = [
  "gas pressure",
  "structural aeroelastic flight high speed aircraft",
  "heat conduction composite slabs",
  "boundary layer control",
  "compressible flow nozzle",
  "combustion chamber injection",
  "laminar turbulent transition",
  "fatigue crack growth",
  "wing tip vortices",
  "propulsion efficiency",
  "sigmoid",
  "local circumferential buckling of thin circular cylindrical shells the problem of circumferential buckling of a thin circular cylindrical shell due to compressive hoop stresses which vary in the axial direction is examined for extremely localised compressive hoop stress distributions resulting from thermal discontinuity effects or from a uniform radial line loading the buckle pattern should also be localised simplified analyses into these two types of problem are considered which show that only a limited number of buckle deflection modes needs to be assumed"
]

In [12]:
# Run TF-IDF queries in batch (print top-5 results for each), using the function you created
def run_batch_tfidf(queries):
    results = {}
    for i, q in enumerate(queries, 1):
        res = tfidf_retrieve(q)
        results[f"Q{i}"] = res
    return results

tfidf_results = run_batch_tfidf(tfidf_queries)

for qid, res in tfidf_results.items():
    print(qid, "=>", res[:5])


Q1 => [['ID 1312', 0.11322513412676355], ['ID 665', 0.11303617510927584], ['ID 337', 0.10743966319327071], ['ID 169', 0.10582766453615478], ['ID 340', 0.10420518640901086]]
Q2 => [['ID 12', 0.1292711170257401], ['ID 875', 0.09807911646438958], ['ID 746', 0.08545615376720098], ['ID 51', 0.07778964622250119], ['ID 884', 0.0770394824056221]]
Q3 => [['ID 399', 0.27718287039546974], ['ID 5', 0.21859333597820912], ['ID 485', 0.20624919698219496], ['ID 144', 0.16773978783721719], ['ID 181', 0.13580305980668986]]
Q4 => [['ID 748', 0.14161595299106106], ['ID 265', 0.13700015810584773], ['ID 451', 0.12609559445032328], ['ID 1288', 0.10889564208337571], ['ID 368', 0.10538189275490609]]
Q5 => [['ID 389', 0.12635126463475174], ['ID 965', 0.11104757044126946], ['ID 118', 0.10622118724932977], ['ID 326', 0.09521521626988949], ['ID 1366', 0.09474812774262026]]
Q6 => [['ID 1143', 0.11931428415992625], ['ID 1254', 0.10439392290792189], ['ID 1241', 0.0964745688298795], ['ID 635', 0.08373182780914523], ['


## Part 1.3 – Conceptual Questions

Answer the following questions:

**1. What is the difference between data retrieval and information retrieval?**
The differnce between data retrieval and information retrieval is that in data retrieval you are looking up specific data using exact queries/methods. While in information retrieval you are interested in finding relevant data/results based on more general queries.

**For the following scenarios, which approach would be suitable data retrieval or information retrieval? Explain your reasoning.** <br>
1.a A clerk in pharmacy uses the following query: Medicine_name = Ibuprofen_400mg

Data retrieval would be suitable here because she is looking for a specific item that the clerk knows should be in the database.

1.b A clerk in pharmacy uses the following query: An anti-biotic medicine 
Information retrieval, anti-biotic medicine will by nature return many results becauase there are a multidtude of anti-biotic medicines. Therefore the clerk would like to find the most relevant results.

1.c Searching for the schedule of a flight using the following query: Flight_ID = ZEFV2
Data retrieval because you are looking for a specific flight with an ID which should only return one accurate result.

1.d Searching an E-commerce website using the following query to find an specific shoe: Brooks Ghost 15
Data retrieval because the user is looking for one specific shoe, however the website may have to convert the name into the relevant shoe or somethign like that-

1.e Searching the same E-commerce website using the following query: Nice running shoes
This is information retrieval because there are multiple relevant documents for this query, (a lot of different shoes).
